# Recreate Simulations with Configurations from Related Work

This notebook reproduces Monte Carlo simulations from key publications in the field of tissue optics and reflectance spectroscopy. By recreating these simulations with their original parameter configurations, we can benchmark our implementation against established results and ensure consistency with published findings.

## Related Work Parameter Configurations

### Parameter Comparison Table

| Study | Layers | μₐ Range [cm⁻¹] | μₛ Range [cm⁻¹] | g Range | n | Wavelength [nm] | Samples |
|-------|--------|-----------------|-----------------|---------|---|-----------------|---------|
| [Yudovsky 2011b](https://doi.org/10.1117/1.3640814) | 2 | 0.1-20 (mm⁻¹) | 5-20 (mm⁻¹, μₛ') | - | 1.40 | NIR | - |
| [Tsui 2018](https://doi.org/10.1364/BOE.9.001531) | 4 | 0.1-350 | 10-1000 | 0.715-0.92 | 1.42 | 410-760 | 30,000 |
| [Wang 2019](https://doi.org/10.3390/photonics6020061) | 3 | 0.1-350 | 10-700 | 0.715-0.835 | - | 401-583 | 57,695 |
| [Lan 2023](https://doi.org/10.1364/BOE.490164) | 1 | 0-10 | 100-350 | 0.8-1.0 | - | 450-650 | 5,000 |
| [Manojlovic 2025](https://doi.org/10.1117/1.JBO.30.1.016004) | 2 | Physiological* | 20-80 @ 500nm | Eq.7 | Eq.6 | 400-1000 | 70,000 |
| **Our Current** | Variable | 0.0012-886.5 | 3.2-4488 | 0.8-0.95 | 1.33-1.54 | - | - |

*Physiological: Based on chromophore concentrations (see detailed table below)

### Sampling Strategies Summary

| Study | Photons | Beam Type | Sampling Method | Special Notes |
|-------|---------|-----------|-----------------|---------------|
| Yudovsky 2011b | - | - | - | Diffusion approximation |
| Tsui 2018 | 100M | Gaussian | Uniform | 3 SDS configurations |
| Wang 2019 | 1B | Gaussian (0.22mm) | Uniform | 7 parameters |
| Lan 2023 | - | - | Latin Hypercube | - |
| Manojlovic 2025 | - | - | Uniform | Split: 50% Bayesian, 50% NN |

In [ ]:
import copy
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from rich.progress import track
from scipy.stats.qmc import LatinHypercube

from mcmlnet.susi.calculate_spectra import (
    calculate_spectrum_for_physical_batch,
)
from mcmlnet.transforms.related_work_transformations import (
    LanConstants,
    ManojlovicConstants,
    ManojlovicTransformation,
    TsuiConstants,
)

load_dotenv()

%matplotlib inline

In [ ]:
# global MCMCL parameters
MCO_FOLDER = ""
IGNORE_A = True

## Resimulate Physical Parameters like Original Publications

In [ ]:
dummy_wvl = 400 * 10**-9


def generate_uniformly_distributed_data(
    n_samples: int, min_val: list[float], max_vals: list[float], seed: int = 42
) -> list[np.ndarray]:
    """
    Generate uniformly distributed data for physical parameters.

    Args:
        n_samples: Number of samples
        min_val: List of minimum values for each parameter
        max_vals: List of maximum values for each parameter
        seed: Random seed

    Returns:
        List of uniformly distributed data for each parameter
    """
    if len(min_val) != len(max_vals):
        raise ValueError("min_val and max_vals must have the same length")

    random_state = np.random.RandomState(seed=seed)
    parameters = [
        random_state.uniform(_min, _max, size=n_samples)
        for _min, _max in zip(min_val, max_vals, strict=False)
    ]

    return parameters


def reorder_parameters(parameter_array: np.ndarray, n_layers: int) -> np.ndarray:
    """
    Reorder parameters to match the simulation DataFrame header
    from grouped n_layer x mu_a, n_layer x us, ... to layer-wise
    mu_a, us, g, n, d, mu_a, us, g, n, d, ... ordering.

    Args:
        parameter_array: Array of physical tissue parameters
        n_layers: Number of tissue layers

    Returns:
        Reordered parameters
    """
    col_ids = np.array([i + j * n_layers for i in range(n_layers) for j in range(5)])
    return parameter_array[:, col_ids]


def generate_simulation_header(
    n_layers: int, wavelengths: float | np.ndarray = dummy_wvl
) -> pd.MultiIndex:
    """Generate a multi-index header for a simulation DataFrame.

    Args:
        n_layers: Number of tissue layers
        wavelengths: Wavelengths for DataFrame header

    Returns:
        Multi-index header for a simulation DataFrame
    """
    rounded_to_picometer_precision = np.round(wavelengths, 12)
    if isinstance(wavelengths, float):
        rounded_to_picometer_precision = [rounded_to_picometer_precision]
    else:
        rounded_to_picometer_precision = rounded_to_picometer_precision.tolist()
    return pd.MultiIndex.from_product(
        [
            [f"layer{i}" for i in range(n_layers)],
            rounded_to_picometer_precision,
            ["ua", "us", "g", "n", "d"],
        ],
        names=["layer [top first]", "wavelength [m]", "parameter"],
    )

### [Tsui 2018](https://doi.org/10.1364/BOE.9.001531)

- **Parameters**:
    - $\mu_{a,1}\in[0.1,5] cm^{-1}$, $\mu_{a,1} = \mu_{a,2}$, $\mu_{a,3}\in[1,350] cm^{-1}$, $\mu_{a,4}\in[0.01,15] cm^{-1}$,
    - $\mu_{s,1}\in[100, 1000] cm^{-1}$, $\mu_{s,2}\in[10,500] cm^{-1}$, $\mu_{s,3} = 1.35\cdot\mu_{s,2}$, $\mu_{s,4}\in[10, 500] cm^{-1}$,
    - $g_1=0.92$, $g_2=g_3=0.75$, $g_4=0.715$,
    - $n=1.42$,
    - $d_1 = d_2\in[5, 30] \mu m$, $d_3\in[10, 60] \mu m$,
    - NIR: in-vivo range from 410-760 nm -> scan range and filter for fitting physiological parameters, then resimulate - or generate grid of physical-to-reflectance map and construct physiological spectra by looking their physical parameters up.
- **Sampling Strategy**: Using a Gaussian beam, 100 Mio photons were used, a total of 30.000 samples were created as pre-simulated database (across 3 SDS's).

In [ ]:
# load parameter ranges
consts_tsui = TsuiConstants()
ranges = [
    consts_tsui.mu_a_1_range,
    consts_tsui.mu_a_3_range,
    consts_tsui.mu_a_4_range,
    consts_tsui.mu_s_1_range,
    consts_tsui.mu_s_2_range,
    consts_tsui.mu_s_4_range,
    consts_tsui.d_1_range,
    consts_tsui.d_2_range,
    consts_tsui.d_3_range,
]
min_vals = [_tuple[0] for _tuple in ranges]
max_vals = [_tuple[1] for _tuple in ranges]
print(f"Min values: {min_vals}")
print(f"Max values: {max_vals}")

# generate random samples
n_samples = consts_tsui.n_samples_sim
n_photons = consts_tsui.n_photons
physical_sample = generate_uniformly_distributed_data(n_samples, min_vals, max_vals)

# create physical parameter set and insert fixed (non-sampled) parameters
mu_a_2 = copy.deepcopy(physical_sample[0])
mu_s_3 = copy.deepcopy(physical_sample[4])
mu_s_3 = np.array([val * 1.35 for val in mu_s_3])
g_1 = np.array([consts_tsui.g_1] * n_samples)
g_2 = np.array([consts_tsui.g_2] * n_samples)
g_3 = np.array([consts_tsui.g_3] * n_samples)
g_4 = np.array([consts_tsui.g_4] * n_samples)
n = np.array([consts_tsui.n] * n_samples)
d_4 = np.array([0.2] * n_samples)  # assume 20 cm thick ("infinite")
physical_sample.insert(1, mu_a_2)
physical_sample.insert(6, mu_s_3)
physical_sample.insert(8, g_1)
physical_sample.insert(9, g_2)
physical_sample.insert(10, g_3)
physical_sample.insert(11, g_4)
# refractive index is constant for all layers
physical_sample.insert(12, n)
physical_sample.insert(13, n)
physical_sample.insert(14, n)
physical_sample.insert(15, n)
# insert deepest d value
physical_sample.insert(19, d_4)
print(f"Physical sample shape: {np.stack(physical_sample).shape}")

In [ ]:
# convert data to pandas dataframe
n_layers = 4
params_tsui = pd.DataFrame(
    reorder_parameters(np.stack(physical_sample, axis=1), n_layers),
    columns=generate_simulation_header(n_layers),
)
print(f"Parameter shape: {params_tsui.shape}")
params_tsui.head()

In [ ]:
# run simulation
save_dir = os.path.join(
    os.getenv("data_dir"), "raw/related_work_reimplemented/tsui_2018_resim.parquet"
)

try:
    params_tsui = pd.read_parquet(save_dir)
except FileNotFoundError:
    reflectance_tsui = calculate_spectrum_for_physical_batch(
        batch=params_tsui,
        wavelengths=np.array([dummy_wvl]),
        nr_photons=n_photons,
        batch_id=str(0),
        mci_base_folder=MCO_FOLDER,
        ignore_a=IGNORE_A,
        mco_file=os.path.join(MCO_FOLDER, "batch.mco"),
    )
    # add reflectance to parameter dataframe
    params_tsui[("reflectance", "", "")] = reflectance_tsui
    # save to csv
    params_tsui.to_parquet(save_dir)

In [ ]:
print(f"Parameter shape: {params_tsui.shape}")
params_tsui.head()

### [Lan 2023](https://doi.org/10.1364/BOE.490164)

- **Parameters**: 5000 simulations from $\mu_a$ ∈ (0, 10) $cm^{−1}$, $\mu_s$ ∈ (100, 350) $cm^{−1}$, $g$ ∈ (0.8, 1.0)
- Phantom from 450-650 nm -> scan range and filter for fitting physiological parameters, then resimulate. Comment: Without specified $n$ in the paper, the simulations are harder to reproduce
- **Sampling Strategy**: Latin Hypercube Sampling

#### Uniform Sampling

In [ ]:
# define parameter ranges
consts_lan = LanConstants()
ranges = [consts_lan.mu_a_range, consts_lan.mu_s_range, consts_lan.g_range]
min_vals = [_tuple[0] for _tuple in ranges]
max_vals = [_tuple[1] for _tuple in ranges]
print(f"Min values: {min_vals}")
print(f"Max values: {max_vals}")

# generate random samples
n_samples = consts_lan.n_samples_sim
n_photons = consts_lan.n_photons  # not specified in original work
physical_sample = generate_uniformly_distributed_data(n_samples, min_vals, max_vals)

# create physical parameter set and insert fixed (non-sampled) parameters
n = np.array([consts_lan.n] * n_samples)
d = np.array([0.2] * n_samples)  # assume 20 cm thick ("infinite")
physical_sample.append(n)
physical_sample.append(d)
print(f"Physical sample shape: {np.stack(physical_sample).shape}")

In [ ]:
# convert data to pandas dataframe
n_layers = 1
params_lan = pd.DataFrame(
    np.stack(physical_sample, axis=1), columns=generate_simulation_header(n_layers)
)
print(f"Parameter shape: {params_lan.shape}")
params_lan.head()

In [ ]:
# run simulation
save_dir = os.path.join(
    os.getenv("data_dir"), "raw/related_work_reimplemented/lan_2023_resim.parquet"
)

try:
    params_lan = pd.read_parquet(save_dir)
except FileNotFoundError:
    reflectance_lan = calculate_spectrum_for_physical_batch(
        batch=params_lan,
        wavelengths=np.array([dummy_wvl]),
        nr_photons=n_photons,
        batch_id=str(0),
        mci_base_folder=MCO_FOLDER,
        ignore_a=IGNORE_A,
        mco_file=os.path.join(MCO_FOLDER, "batch.mco"),
    )
    # add reflectance to parameter dataframe
    params_lan[("reflectance", "", "")] = reflectance_lan
    # save to csv
    params_lan.to_parquet(save_dir)

In [ ]:
print(f"Parameter shape: {params_lan.shape}")
params_lan.head()

#### Latin Hypercube Sampling

In [ ]:
# repeat with LHS sampling - generate and scale latin hypercube samples
lhs_sampler = LatinHypercube(d=3)
lhs_samples = lhs_sampler.random(n_samples)
lhs_samples = lhs_samples * (np.array(max_vals) - np.array(min_vals)) + np.array(
    min_vals
)

# insert fixed parameters
n = np.array([consts_lan.n] * n_samples)
d = np.array([0.2] * n_samples)  # assume 20 cm thick ("infinite")
concatenated = np.concatenate([lhs_samples, n[:, None], d[:, None]], axis=1)

In [ ]:
# convert data to pandas dataframe
n_layers = 1
params_lan_lhs = pd.DataFrame(
    concatenated, columns=generate_simulation_header(n_layers)
)
print(f"Parameter shape: {params_lan_lhs.shape}")
params_lan_lhs.head()

In [ ]:
# run simulation
save_dir = os.path.join(
    os.getenv("data_dir"), "raw/related_work_reimplemented/lan_lhs_2023_resim.parquet"
)

try:
    params_lan_lhs = pd.read_parquet(save_dir)
except FileNotFoundError:
    reflectance_lan_lhs = calculate_spectrum_for_physical_batch(
        batch=params_lan_lhs,
        wavelengths=np.array([dummy_wvl]),
        nr_photons=n_photons,
        batch_id=str(0),
        mci_base_folder=MCO_FOLDER,
        ignore_a=IGNORE_A,
        mco_file=os.path.join(MCO_FOLDER, "batch.mco"),
    )
    # add reflectance to parameter dataframe
    params_lan_lhs[("reflectance", "", "")] = reflectance_lan_lhs
    # save to csv
    params_lan_lhs.to_parquet(save_dir)

In [ ]:
print(f"Parameter shape: {params_lan_lhs.shape}")
params_lan_lhs.head()

### [Manojlovic 2025](https://doi.org/10.1117/1.JBO.30.1.016004)

- **Parameters**: See Eq.1 for absorption of epidermis, Eq.4 for absorption of dermis, Eq.6 for the refractive index, and Eq.7 for the anisotropy. Scattering coefficients and other parameters are defined as following: $b_{Mie} = 1.2$, $f_{Ray} = 10^{-7}$, $d_{epi} = 0.01 cm$, and $d_{der} = 1 cm$. Chromophore ranges:

| Chromophore | Symbol | Range | Units |
|-------------|--------|-------|-------|
| Melanin volume fraction | fₘ | 0.001-0.05 | - |
| Deoxyhemoglobin volume fraction | f_Hb | 0.001-0.05 | - |
| Oxyhemoglobin volume fraction | f_HbO₂ | 0.001-0.05 | - |
| Bilirubin concentration | f_brub | 10⁻⁷-0.1 | mM |
| Reduced cytochrome C oxidase | f_CO | 10⁻⁷-2 | mM |
| Oxidized cytochrome C oxidase | f_COO₂ | 10⁻⁷-2 | mM |
| Scattering at 500nm | a | 20-80 | cm⁻¹ |

- **Sampling Strategy**: "Therefore, we simulated 70,000 spectra using the model described in Sec. 2.1, of which one half was used for Bayesian inference, and the other half was used to train the F-NN."

In [ ]:
# define parameter ranges
consts_manoj = ManojlovicConstants()
ranges = [
    consts_manoj.f_mel_range,
    consts_manoj.f_hb_range,
    consts_manoj.f_hbo2_range,
    consts_manoj.f_brub_range,
    consts_manoj.f_co_range,
    consts_manoj.f_coo2_range,
    consts_manoj.a_mie_range,
]
min_vals = [_tuple[0] for _tuple in ranges]
max_vals = [_tuple[1] for _tuple in ranges]
print(min_vals, max_vals)


manojlovic_physical = ManojlovicTransformation(
    torch.from_numpy(consts_manoj.wavelengths * 10**9)
)
# generate random samples
n_samples = consts_manoj.n_samples_sim
n_photons = consts_manoj.n_photons

# generate random samples
physiological_sample = generate_uniformly_distributed_data(
    n_samples, min_vals, max_vals
)
manoj_physio_df = pd.DataFrame(
    np.stack(physiological_sample, axis=-1),
    columns=["f_mel", "f_hb", "f_hbo2", "f_brub", "f_co", "f_coo2", "a"],
)
print(f"Physio shape: {manoj_physio_df.shape}")


def _col_to_tensor(col: str) -> torch.Tensor:
    return torch.from_numpy(manoj_physio_df[f"{col}"].to_numpy()).float()


# convert to physical parameters
mu_a_epi = manojlovic_physical.mu_a_epi(_col_to_tensor("f_mel"))
physical_sample = torch.stack(
    [
        mu_a_epi,
        manojlovic_physical.scattering(
            _col_to_tensor("a"),
            torch.tensor([consts_manoj.f_ray]),
            torch.tensor([consts_manoj.b_mie]),
        ),
        manojlovic_physical.anisotropy()[None, :].repeat(len(mu_a_epi), 1),
        manojlovic_physical.refractive_index()[None, :].repeat(len(mu_a_epi), 1),
        torch.ones_like(mu_a_epi) * consts_manoj.d_epi,
        manojlovic_physical.mu_a_dermis(
            _col_to_tensor("f_hb"),
            _col_to_tensor("f_hbo2"),
            _col_to_tensor("f_brub"),
            _col_to_tensor("f_co"),
            _col_to_tensor("f_coo2"),
        ),
        manojlovic_physical.scattering(
            _col_to_tensor("a"),
            torch.tensor([consts_manoj.f_ray]),
            torch.tensor([consts_manoj.b_mie]),
        ),
        manojlovic_physical.anisotropy()[None, :].repeat(len(mu_a_epi), 1),
        manojlovic_physical.refractive_index()[None, :].repeat(len(mu_a_epi), 1),
        # NOTE: This can be a problem! A lot of photons will be lost
        # at the lower end if the dermis is too thin!
        torch.ones_like(mu_a_epi) * consts_manoj.d_dermis * 10,
    ],
    dim=-1,
)
print(f"Physical sample shape: {physical_sample.shape}")

In [ ]:
# reshape and re-order the parameter columns manually
top_layer = physical_sample[:, :, :5].reshape(len(physical_sample), -1)
bottom_layer = physical_sample[:, :, 5:].reshape(len(physical_sample), -1)
concatenated = torch.cat([top_layer, bottom_layer], dim=1).numpy()

# convert data to pandas dataframe
n_layers = 2
wavelengths = consts_manoj.wavelengths
params_manoj = pd.DataFrame(
    concatenated, columns=generate_simulation_header(n_layers, wavelengths)
)
print(f"Parameter shape: {params_manoj.shape}")
params_manoj.head()

In [ ]:
# run simulation
save_dir = os.path.join(
    os.getenv("data_dir"),
    "raw/related_work_reimplemented/manojlovic_2025_resim_thick.parquet",
)

batch_size = 1000
results = []

if n_samples % batch_size != 0:
    iterator = track(range(n_samples // batch_size + 1))
else:
    iterator = track(range(n_samples // batch_size))

try:
    result_df = pd.read_parquet(save_dir)
except FileNotFoundError:
    for i in iterator:
        _reflectance = calculate_spectrum_for_physical_batch(
            batch=params_manoj.iloc[i * batch_size : (i + 1) * batch_size],
            wavelengths=consts_manoj.wavelengths,
            nr_photons=n_photons,
            batch_id=str(i),
            mci_base_folder=MCO_FOLDER,
            ignore_a=IGNORE_A,
            mco_file=os.path.join(MCO_FOLDER, "batch.mco"),
        )
        # save physiological parameters to save space
        _df = manoj_physio_df.iloc[i * batch_size : (i + 1) * batch_size].copy()
        # add reflectance to copy of parameter dataframe
        _df = pd.concat(
            [_df.reset_index(drop=True), _reflectance.reset_index(drop=True)], axis=1
        )
        results.append(_df)
    result_df = pd.concat(results).reset_index(drop=True)
    result_df.to_parquet(save_dir)

In [ ]:
print(f"Parameter shape: {result_df.shape}")
params_manoj.head()

#### Plot Transformation Extinction Coefficients for Manojlovic

In [ ]:
plt.figure(figsize=(10, 5))
_wavelengths_manoj = manojlovic_physical.wavelengths
# Get the extinction coefficients for the Manojlovic model
hbo2 = manojlovic_physical.eHbO2.numpy()
hb = manojlovic_physical.eHb.numpy()
brub = manojlovic_physical.eBrub.numpy() / 1000  # 1/mM to 1/M
coo2 = manojlovic_physical.e_cyto_ox.numpy() / 1000  # 1/mM to 1/M
co = manojlovic_physical.e_cyto_red.numpy() / 1000  # 1/mM to 1/M
# NOTE: The following are regular absorption coefficients, not extinction coefficients
mel = manojlovic_physical._melanin_absorption(torch.Tensor([0.01])).numpy()
base = manojlovic_physical._base_absorption().numpy()
plt.plot(_wavelengths_manoj, hbo2, label="HbO2")
plt.plot(_wavelengths_manoj, hb, label="Hb")
plt.plot(_wavelengths_manoj, brub, label="Brub")
plt.plot(_wavelengths_manoj, coo2, label="CytoOx")
plt.plot(_wavelengths_manoj, co, label="CytoRed")
plt.plot(_wavelengths_manoj, mel, label="Melanin")
plt.plot(_wavelengths_manoj, base, label="Base")
plt.vlines(970, 1, 1e8, color="black", linestyle="--", label="970 nm Cut-off")
plt.legend(ncol=2)
plt.yscale("log")
plt.grid()
plt.title("Transformation Coefficients")
plt.xlabel("Wavelength [nm]")
plt.ylabel(r"Extinction Coefficients [m$^{-1}$/M]")
plt.show()

### Plot Transformation Maximum Absorption Coefficients for Manojlovic 

In [ ]:
vhb_max = 0.05
hb_pre_factor = (
    np.log(10) * manojlovic_physical.cHb / manojlovic_physical.gmw_hb * vhb_max
)

plt.figure(figsize=(10, 5))
_wavelengths_manoj = manojlovic_physical.wavelengths
# Get the absorption coefficients for the Manojlovic model
ua_hbo2 = manojlovic_physical.eHbO2.numpy() * hb_pre_factor
ua_hb = manojlovic_physical.eHb.numpy() * hb_pre_factor
ua_brub = manojlovic_physical.eBrub.numpy() * 0.1 / 1000  # 1/mM to 1/M
ua_coo2 = manojlovic_physical.e_cyto_ox.numpy() * 2 / 1000  # 1/mM to 1/M
ua_co = manojlovic_physical.e_cyto_red.numpy() * 2 / 1000  # 1/mM to 1/M
mel = manojlovic_physical._melanin_absorption(torch.Tensor([0.05])).numpy()
base = manojlovic_physical._base_absorption().numpy()
# Get maximal epidermis and dermis absorption coefficients
ua_epidermis = manojlovic_physical.mu_a_epi(torch.Tensor([0.05])).numpy()
ua_dermis = manojlovic_physical.mu_a_dermis(
    torch.Tensor([0.05]),
    torch.Tensor([0.05]),
    torch.Tensor([0.1]),
    torch.Tensor([2]),
    torch.Tensor([2]),
).numpy()
plt.plot(_wavelengths_manoj, ua_hbo2, label="HbO2")
plt.plot(_wavelengths_manoj, ua_hb, label="Hb")
plt.plot(_wavelengths_manoj, ua_brub, label="Brub")
plt.plot(_wavelengths_manoj, ua_coo2, label="CytoOx")
plt.plot(_wavelengths_manoj, ua_co, label="CytoRed")
plt.plot(_wavelengths_manoj, mel, label="Melanin")
plt.plot(_wavelengths_manoj, base, label="Base")
plt.plot(_wavelengths_manoj, ua_epidermis, label="Epidermis", linestyle="--")
plt.plot(_wavelengths_manoj, ua_dermis, label="Dermis", linestyle="--")
plt.vlines(970, 1, 1e6, color="black", linestyle="--", label="970 nm Cut-off")
plt.legend(ncol=2)
plt.yscale("log")
plt.grid()
plt.title("Transformation Coefficients")
plt.xlabel("Wavelength [nm]")
plt.ylabel("Absorption Coefficient [m$^{-1}$]")
plt.show()

#### Plot Example Reflectances for Manojlovic

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(
    _wavelengths_manoj, result_df.drop(columns=["Unnamed: 0"]).iloc[:5, 7:].to_numpy().T
)
plt.vlines(970, 0, 0.25, color="black", linestyle="--", label="970 nm Cut-off")
plt.legend()
plt.title("Reflectance")
plt.xlabel("Wavelength [nm]")
plt.ylabel("Reflectance")
plt.grid()
plt.show()